<a href="https://colab.research.google.com/github/mbmanoj904-hue/Generative-ai/blob/main/Langchain_Lmm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ============================================================
# LANGGRAPH + LLM + RAG PROJECT
# Intelligent Research Assistant
# Everything in ONE Colab cell
# ============================================================

!pip install -q -U langgraph langchain langchain-community \
langchain-text-splitters langchain-huggingface faiss-cpu \
sentence-transformers openai

# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

import os
from google.colab import userdata

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List

from openai import OpenAI


# ------------------------------------------------------------
# 2. API KEY
# ------------------------------------------------------------

# Add Groq API Key in:
# Colab → Secrets → Groq
try:
    api_key = userdata.get("Groq")
except Exception:
    api_key = None

if not api_key:
    raise ValueError(
        "Groq API key not found in Colab Secrets under the name 'Groq'."
    )

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key
)


# ------------------------------------------------------------
# 3. SAMPLE KNOWLEDGE BASE
# ------------------------------------------------------------

documents = [
    Document(page_content="""
    Artificial Intelligence (AI) is the field of computer science
    concerned with creating systems that can perform tasks that
    normally require human intelligence. These tasks include
    learning, reasoning, perception, decision making and language
    understanding.
    """),

    Document(page_content="""
    Machine Learning is a subset of Artificial Intelligence.
    Machine learning systems learn patterns from data and use those
    patterns to make predictions or decisions without being
    explicitly programmed for every individual task.
    """),

    Document(page_content="""
    Deep Learning is a subset of machine learning that uses
    artificial neural networks with multiple layers. Deep learning
    is widely used for image recognition, speech recognition,
    natural language processing and recommendation systems.
    """),

    Document(page_content="""
    Large Language Models (LLMs) are machine learning models
    trained on large amounts of text. They can understand and
    generate natural language. Examples of applications include
    question answering, summarization, coding assistance and
    content generation.
    """),

    Document(page_content="""
    Retrieval Augmented Generation (RAG) combines information
    retrieval with a language model. Relevant documents are first
    retrieved from a knowledge base and then provided to the LLM
    so that the model can generate an answer based on the retrieved
    information.
    """),

    Document(page_content="""
    LangGraph is a framework for building stateful, multi-step
    applications with language models. It represents an application
    as a graph containing nodes and edges. Nodes perform operations
    and edges control the flow between those operations.
    """),

    Document(page_content="""
    Vector databases store numerical representations called
    embeddings. Embeddings represent the semantic meaning of text.
    A similarity search can then retrieve documents that are
    semantically related to a user's question.
    """)
]


# ------------------------------------------------------------
# 4. SPLIT DOCUMENTS
# ------------------------------------------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Documents:", len(documents))
print("Chunks:", len(chunks))


# ------------------------------------------------------------
# 5. CREATE EMBEDDINGS
# ------------------------------------------------------------

print("\nCreating embeddings...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# ------------------------------------------------------------
# 6. CREATE FAISS VECTOR DATABASE
# ------------------------------------------------------------

vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector database created.")


# ------------------------------------------------------------
# 7. LANGGRAPH STATE
# ------------------------------------------------------------

class GraphState(TypedDict):
    question: str
    context: str
    analysis: str
    answer: str
    validation: str


# ------------------------------------------------------------
# 8. LLM FUNCTION
# ------------------------------------------------------------

def call_llm(prompt):

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


# ------------------------------------------------------------
# 9. NODE 1 - QUERY ANALYZER
# ------------------------------------------------------------

def analyze_query(state: GraphState):

    question = state["question"]

    prompt = f"""
You are an AI research assistant.

Analyze the user's question and identify:
1. The main topic
2. What information is required
3. What should be checked before answering

Do not give the final answer yet.

Question:
{question}
"""

    analysis = call_llm(prompt)

    print("\n========== QUERY ANALYSIS ==========")
    print(analysis)

    return {
        "analysis": analysis
    }


# ------------------------------------------------------------
# 10. NODE 2 - RETRIEVER
# ------------------------------------------------------------

def retrieve_information(state: GraphState):

    question = state["question"]

    results = vector_db.similarity_search(
        question,
        k=3
    )

    context = "\n\n".join(
        [doc.page_content for doc in results]
    )

    print("\n========== RETRIEVED INFORMATION ==========")
    print(context)

    return {
        "context": context
    }


# ------------------------------------------------------------
# 11. NODE 3 - REASONING / ANSWER GENERATION
# ------------------------------------------------------------

def generate_answer(state: GraphState):

    question = state["question"]
    analysis = state["analysis"]
    context = state["context"]

    prompt = f"""
You are a helpful AI research assistant.

Use the retrieved context to answer the question.

Question:
{question}

Query analysis:
{analysis}

Retrieved context:
{context}

Instructions:
- Answer clearly and accurately.
- Use only information supported by the context.
- If the context does not contain enough information,
  say that the available knowledge base is insufficient.
- Give a concise explanation.
- Do not reveal private chain-of-thought.
- Provide the useful conclusion, not hidden reasoning.

Answer:
"""

    answer = call_llm(prompt)

    print("\n========== GENERATED ANSWER ==========")
    print(answer)

    return {
        "answer": answer
    }


# ------------------------------------------------------------
# 12. NODE 4 - VALIDATION
# ------------------------------------------------------------

def validate_answer(state: GraphState):

    question = state["question"]
    context = state["context"]
    answer = state["answer"]

    prompt = f"""
You are an answer validation system.

Check whether the answer is supported by the provided context.

Question:
{question}

Context:
{context}

Answer:
{answer}

Return:
VALID - if the answer is supported.
INVALID - if important claims are unsupported.

Then give one short reason.
"""

    validation = call_llm(prompt)

    print("\n========== VALIDATION ==========")
    print(validation)

    return {
        "validation": validation
    }


# ------------------------------------------------------------
# 13. BUILD LANGGRAPH
# ------------------------------------------------------------

graph = StateGraph(GraphState)

graph.add_node("analyze", analyze_query)
graph.add_node("retrieve", retrieve_information)
graph.add_node("generate", generate_answer)
graph.add_node("validate", validate_answer)

graph.add_edge(START, "analyze")
graph.add_edge("analyze", "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "validate")
graph.add_edge("validate", END)

app = graph.compile()


# ------------------------------------------------------------
# 14. USER QUERY
# ------------------------------------------------------------

question = input(
    "\nEnter your question: "
)


# ------------------------------------------------------------
# 15. RUN LANGGRAPH
# ------------------------------------------------------------

result = app.invoke({
    "question": question,
    "context": "",
    "analysis": "",
    "answer": "",
    "validation": ""
})


# ------------------------------------------------------------
# 16. FINAL RESULT
# ------------------------------------------------------------

print("\n\n===============================================")
print("                FINAL RESULT")
print("===============================================")

print("\nQUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nVALIDATION:")
print(result["validation"])

print("\n===============================================")
print("LangGraph workflow completed successfully.")
print("===============================================")

Documents: 7
Chunks: 7

Creating embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS vector database created.

Enter your question: what is ai&ml

========== QUERY ANALYSIS ==========
**1. Main topic**  
The question is about the definition and explanation of **Artificial Intelligence (AI) and Machine Learning (ML)** – what these terms mean, how they relate, and what they encompass.

**2. Information required**  
- Clear, concise definitions of AI and ML.  
- A brief overview of the relationship between the two (e.g., ML as a sub‑field of AI).  
- Typical examples or applications that illustrate each concept.  
- (Optional, depending on the user’s needs) A high‑level distinction between AI, ML, and related fields such as deep learning, data science, and robotics.

**3. What should be checked before answering**  
- **Audience level**: Determine whether the user is a layperson, student, or more technical audience so the depth and terminology can be adjusted appropriately.  
- **Scope & ambiguity**: Confirm that “ai&ml” refers to the general concepts of Artificial I